<a href="https://colab.research.google.com/github/mhowlin-web/TP_RAG_ARCA/blob/main/04_generacion_embeddings_arca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP RAG y Agentes

## Asistente para consultas sobre trámites de Monotributo en ARCA

En este notebook voy a transformar cada chunk del corpus de ARCA en un
vector numérico llamado embedding.

El embedding representa semánticamente el contenido del chunk.

Esto me permitirá comparar posteriormente una pregunta con los chunks
del corpus utilizando similitud vectorial.

En este notebook todavía no voy a utilizar Pinecone.

Primero voy a generar y guardar los embeddings. En el próximo notebook
voy a cargar estos vectores en Pinecone.

## Montaje de Google Drive

En esta celda monto Google Drive para acceder al corpus de chunks que
generé en el Notebook 03 y para guardar los embeddings generados.

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Definición de las rutas

En esta celda defino la carpeta del proyecto y la carpeta donde estoy
guardando el corpus.

Utilizo la misma estructura de carpetas que en los notebooks anteriores
para mantener todo el proyecto organizado.

In [4]:
from pathlib import Path

CARPETA_PROYECTO = Path(
    "/content/drive/MyDrive/TP_RAG_ARCA"
)

CARPETA_CORPUS = (
    CARPETA_PROYECTO / "corpus"
)

print("Carpeta del proyecto:")
print(CARPETA_PROYECTO)

print("\nCarpeta del corpus:")
print(CARPETA_CORPUS)

Carpeta del proyecto:
/content/drive/MyDrive/TP_RAG_ARCA

Carpeta del corpus:
/content/drive/MyDrive/TP_RAG_ARCA/corpus


## Definición de los archivos

En esta celda defino el archivo de entrada y el archivo de salida.

Como entrada utilizo el corpus dividido en chunks que generé en el
Notebook 03.

Como salida voy a generar un archivo JSON que contiene cada chunk,
sus metadatos y su embedding.

In [5]:
ARCHIVO_ENTRADA = (
    CARPETA_CORPUS /
    "corpus_arca_monotributo_chunks.json"
)

ARCHIVO_SALIDA = (
    CARPETA_CORPUS /
    "corpus_arca_monotributo_embeddings.json"
)

print("Archivo de entrada:")
print(ARCHIVO_ENTRADA)

print("\nArchivo de salida:")
print(ARCHIVO_SALIDA)

Archivo de entrada:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo_chunks.json

Archivo de salida:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo_embeddings.json


## Verificación del archivo de entrada

En esta celda verifico que el archivo generado en el Notebook 03 exista
antes de continuar.

In [6]:
if not ARCHIVO_ENTRADA.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo:\n{ARCHIVO_ENTRADA}"
    )

print("Archivo encontrado correctamente.")
print(
    f"Tamaño: {ARCHIVO_ENTRADA.stat().st_size:,} bytes"
)

Archivo encontrado correctamente.
Tamaño: 43,723 bytes


## Instalación de la librería de embeddings

En esta celda instalo `sentence-transformers`.

Voy a utilizar un modelo de Hugging Face especializado en generar
representaciones vectoriales de textos.

Utilizo el mismo modelo que ya había utilizado anteriormente en el
tutorial:

`sentence-transformers/all-MiniLM-L6-v2`

In [7]:
!pip install -q sentence-transformers

## Importación de librerías

En esta celda importo las librerías que voy a utilizar.

Utilizo `json` para cargar y guardar el corpus y `SentenceTransformer`
para generar los embeddings.

In [8]:
import json

from sentence_transformers import SentenceTransformer

## Carga del corpus de chunks

En esta celda cargo el archivo generado en el Notebook 03.

Cada elemento representa un chunk y contiene tanto su texto como los
metadatos necesarios para identificar su documento de origen.

In [9]:
with open(
    ARCHIVO_ENTRADA,
    "r",
    encoding="utf-8"
) as archivo:

    chunks = json.load(archivo)

print(
    f"Chunks cargados: {len(chunks)}"
)

Chunks cargados: 43


## Verificación de los chunks

En esta celda verifico algunos datos básicos del corpus antes de generar
los embeddings.

Quiero asegurarme de que estoy trabajando con los 43 chunks generados
en el notebook anterior y que cada uno tiene texto.

In [10]:
print("=" * 80)
print("VERIFICACIÓN DEL CORPUS")
print("=" * 80)

print(
    f"Cantidad de chunks: {len(chunks)}"
)

chunks_sin_texto = [
    chunk
    for chunk in chunks
    if not chunk.get("texto", "").strip()
]

print(
    f"Chunks sin texto: {len(chunks_sin_texto)}"
)

if not chunks:
    raise ValueError(
        "El corpus no contiene chunks."
    )

if chunks_sin_texto:
    raise ValueError(
        "Hay chunks sin texto."
    )

VERIFICACIÓN DEL CORPUS
Cantidad de chunks: 43
Chunks sin texto: 0


## Definición del modelo de embeddings

En esta celda cargo el modelo `all-MiniLM-L6-v2`.

Este modelo transforma cada texto en un vector numérico.

Voy a utilizar exactamente el mismo modelo para los documentos y,
posteriormente, para las preguntas.

Esto es importante porque los vectores deben pertenecer al mismo
espacio semántico para poder compararlos.

In [11]:
NOMBRE_MODELO = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Cargando modelo de embeddings...")

modelo_embeddings = SentenceTransformer(
    NOMBRE_MODELO
)

print("Modelo cargado correctamente.")

Cargando modelo de embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado correctamente.


## Prueba del modelo

En esta celda genero un embedding para un texto de prueba.

Además de comprobar que el modelo funciona, verifico la dimensión
del vector obtenido.

In [12]:
texto_prueba = (
    "¿Cómo puedo obtener la clave fiscal?"
)

embedding_prueba = modelo_embeddings.encode(
    texto_prueba
)

print(
    f"Tipo: {type(embedding_prueba)}"
)

print(
    f"Dimensión del embedding: "
    f"{len(embedding_prueba)}"
)

print("\nPrimeros valores:")

print(
    embedding_prueba[:10]
)

Tipo: <class 'numpy.ndarray'>
Dimensión del embedding: 384

Primeros valores:
[-0.02573639  0.01869446 -0.07980287 -0.02223556 -0.01502143 -0.00536057
  0.07897344  0.03778169  0.02678681  0.02838603]


## Generación de los embeddings

En esta celda genero un embedding para cada chunk del corpus.

Utilizo el texto del chunk como entrada del modelo.

Después convierto el vector a una lista para poder guardarlo fácilmente
en formato JSON.

Conservo todos los metadatos originales y agrego el embedding.

In [13]:
for i, chunk in enumerate(chunks):

    texto = chunk["texto"]

    embedding = modelo_embeddings.encode(
        texto
    )

    chunk["embedding"] = embedding.tolist()

    if (i + 1) % 10 == 0:
        print(
            f"Procesados: {i + 1}/{len(chunks)}"
        )

print(
    f"\nEmbeddings generados: {len(chunks)}"
)

Procesados: 10/43
Procesados: 20/43
Procesados: 30/43
Procesados: 40/43

Embeddings generados: 43


## Verificación de los embeddings

En esta celda verifico que cada chunk tenga un embedding y que todos
los embeddings tengan la misma dimensión.

Todos los vectores deben tener la misma cantidad de componentes porque
posteriormente los voy a almacenar en un índice vectorial de Pinecone.

In [14]:
dimensiones = set()

for chunk in chunks:

    embedding = chunk.get("embedding")

    if embedding is None:
        raise ValueError(
            f"El chunk {chunk.get('chunk_id')} "
            "no tiene embedding."
        )

    dimensiones.add(
        len(embedding)
    )

print(
    f"Dimensiones encontradas: {dimensiones}"
)

if len(dimensiones) != 1:
    raise ValueError(
        "Los embeddings no tienen todos "
        "la misma dimensión."
    )

print(
    "\nTodos los embeddings tienen "
    "la misma dimensión."
)

Dimensiones encontradas: {384}

Todos los embeddings tienen la misma dimensión.


## Inspección de un embedding

En esta celda inspecciono uno de los embeddings generados.

No interpreto individualmente cada número del vector.

Lo importante para esta etapa es verificar que el embedding exista,
tenga la dimensión esperada y esté asociado correctamente con su chunk.

In [15]:
chunk = chunks[0]

print("=" * 80)
print("CHUNK")
print("=" * 80)

print(
    chunk["texto"]
)

print("\n")
print("=" * 80)
print("INFORMACIÓN DEL EMBEDDING")
print("=" * 80)

print(
    f"Dimensión: {len(chunk['embedding'])}"
)

print(
    f"Primeros 10 valores:"
)

print(
    chunk["embedding"][:10]
)

CHUNK
Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegación y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la información sobre cómo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
Menú de contenidos
Qué es
INSCRIPCIÓN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal Electrónico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
Parámetros
Jubilación
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÉS DEL ALTA
Desarrollo de la actividad
Facturación
Pagos
Recategorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen general
Ayuda
Inicio
El primer pas


INFORMACIÓN DEL EMBEDDING
Dimensión: 384
Primeros 10 valores:
[0.010988332331180573, -0.03515464439988136, -0.051754482090473175, -0.09305208921432495, -0.07802926748991013, 0.013049801811575

## Guardado de los embeddings

En esta celda guardo los chunks junto con sus embeddings en un archivo
JSON.

Este archivo representa el resultado de esta etapa del pipeline.

En el siguiente notebook voy a utilizarlo como entrada para cargar
los vectores en Pinecone.

In [16]:
with open(
    ARCHIVO_SALIDA,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        chunks,
        archivo,
        ensure_ascii=False,
        indent=2
    )

print(
    "Archivo guardado correctamente:"
)

print(
    ARCHIVO_SALIDA
)

Archivo guardado correctamente:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo_embeddings.json


## Verificación del archivo generado

En esta celda vuelvo a cargar el archivo de embeddings para verificar
que se haya guardado correctamente.

También compruebo que la cantidad de chunks coincida.

In [17]:
with open(
    ARCHIVO_SALIDA,
    "r",
    encoding="utf-8"
) as archivo:

    chunks_verificados = json.load(
        archivo
    )

print(
    f"Chunks originales: "
    f"{len(chunks)}"
)

print(
    f"Chunks guardados: "
    f"{len(chunks_verificados)}"
)

if len(chunks) != len(chunks_verificados):

    raise ValueError(
        "La cantidad de chunks no coincide."
    )

print("\nVerificación OK.")

Chunks originales: 43
Chunks guardados: 43

Verificación OK.


## Resumen del proceso

En este notebook transformé cada chunk del corpus de ARCA en un embedding.

El proceso que realicé fue:

1. Cargué el corpus de chunks.
2. Cargué el modelo `all-MiniLM-L6-v2`.
3. Generé un vector para cada chunk.
4. Verifiqué que todos los vectores tengan la misma dimensión.
5. Guardé los embeddings junto con los metadatos originales.

El resultado quedó almacenado en:

`corpus_arca_monotributo_embeddings.json`

En el próximo notebook voy a cargar estos embeddings en Pinecone.

A partir de ese momento voy a poder realizar búsquedas semánticas sobre
el corpus de trámites de ARCA.